# PhysioNet ECG image digitization: inference notebook

Turns the ECG images in `test/` into 12-lead signals and writes `submission.csv`. Nothing is trained here. All networks are pretrained and come from two public Kaggle datasets, so the notebook runs with internet off.

The score is the signal-to-noise ratio between the true and the digitised signal, after a time shift of up to 0.2 s and a constant offset are removed. On the rectified page 1 px is about 0.013 mV, so two things matter: bring every page into the same undistorted frame, and locate the trace in that frame with sub-pixel accuracy.

| Stage | Task | Network | Source |
|---|---|---|---|
| 0 | find the page orientation and 9 reference points, warp the page onto a fixed frame | ResNet18d U-Net | hengck23 |
| 1 | find the printed grid, rectify the page to 1700 x 2200 px | ResNet34 U-Net | hengck23 |
| 2 | segment the trace with 6 networks and horizontal flip TTA | EfficientNet U-Nets | Takashi Someya, 2nd place |
| 3 | soft centroid per column, conversion to mV, resampling, split into leads | none | this notebook |

Stages 0 and 1 use the code and weights of `hengck23-demo-submit-physionet` unchanged. The stage 2 networks are re-implemented in plain PyTorch and timm, because the original code needs `segmentation_models_pytorch`, which cannot be installed without internet. The re-implementation loads the released checkpoints with `strict=True` and was checked against the original implementation.

Settings: GPU accelerator (2 x T4 if offered, one GPU takes about twice as long), internet off. Inputs to attach:

- the competition data `physionet-ecg-image-digitization`
- the dataset `hengck23/hengck23-demo-submit-physionet` (stage 0/1 code and weights)
- the dataset `takashisomeya/physionet-final-submission-models` (six stage 2 checkpoints)

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from collections import Counter
from itertools import islice
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
from IPython.display import display
from scipy.signal import resample

T_START = time.time()       # Kaggle's 9 h limit counts from here; deadlines below are relative to T_START


def elapsed():
    return time.time() - T_START


print(f'python {sys.version.split()[0]} | torch {torch.__version__} | timm {timm.__version__} | '
      f'opencv {cv2.__version__} | numpy {np.__version__} | pandas {pd.__version__}')


## Configuration

The full ensemble costs about 46 TFLOPs per ECG and the run has to finish within 9 hours, so the notebook works against a deadline. If the full ensemble does not fit into the time left, a cheaper ensemble is used for some images (see the level table below). Nothing is started after `hard_deadline`; ECGs that are still open then get a zero signal.

In [ ]:
CONFIG = dict(
    competition_dir=None,       # None: search /kaggle/input
    work_dir='/kaggle/working/ecg_infer',
    submission_path='/kaggle/working/submission.csv',
    soft_deadline=7.6 * 3600,   # seconds after T_START; the ensemble level is planned to finish by then
    hard_deadline=8.2 * 3600,   # no new image is started after this
    n_canary=3,                 # training ECGs scored first as a sanity check
    # debugging options
    max_images=None,            # only the first n test ECGs
    min_level=0,                # 3 = cheapest ensemble only
    allow_cpu=False,            # very slow
)


## Inputs

The mount path of an input depends on the Kaggle image (`/kaggle/input/<name>`, `/kaggle/input/competitions/<name>` or `/kaggle/input/datasets/<owner>/<name>`), so each input is found by the files it contains.

In [ ]:
WORK = Path(CONFIG['work_dir'])
WORK.mkdir(parents=True, exist_ok=True)

INPUT_ROOT = Path('/kaggle/input')
STAGE01_WEIGHTS = ['stage0-last.checkpoint.pth', 'stage1-last.checkpoint.pth']
STAGE2_WEIGHTS = ['whole_b7_lb22.93.pth', 'whole_v2_l_lb22.60.pth', 'series_b6_shared_conv2d_lb23.10.pth',
                  'series_b6_shared_conv2d_lb23.00.pth', 'series_v2_l_conv3d_lb22.92.pth',
                  'series_v2_l_conv2d_lb22.85.pth']


def locate(markers, root=INPUT_ROOT, max_depth=4, hint=''):
    """Folder below `root` that contains all `markers`.

    The match closest to the root wins, ties go to a path containing `hint`.
    The big train/ and test/ folders are never entered.
    """
    found, queue = [], [(Path(root), 0)]
    while queue:
        folder, depth = queue.pop(0)
        try:
            entries = {e.name: e.is_dir() for e in os.scandir(folder)}
        except OSError:
            continue
        if all(m in entries for m in markers):
            found.append(folder)
        elif depth < max_depth:
            queue += [(folder / name, depth + 1) for name, is_dir in sorted(entries.items())
                      if is_dir and name not in ('train', 'test')]
    return min(found, key=lambda p: (len(p.parts), hint not in str(p).lower()), default=None)


def show_inputs(folder=INPUT_ROOT, depth=0, max_depth=3):
    """Print what is mounted below /kaggle/input, to see what is missing when an input is not found."""
    try:
        entries = sorted(os.scandir(folder), key=lambda e: e.name)
    except OSError:
        return
    for e in entries[:30]:
        print('    ' * depth + e.name + ('/' if e.is_dir() else ''))
        if e.is_dir() and depth < max_depth and e.name not in ('train', 'test'):
            show_inputs(e.path, depth + 1, max_depth)


COMP = Path(CONFIG['competition_dir']) if CONFIG['competition_dir'] else locate(['test.csv', 'test'], hint='ecg')
HENGCK = locate(['stage1_model.py', 'weight'], hint='hengck')
MODELS = locate([STAGE2_WEIGHTS[0]], hint='physionet')

missing = []
if COMP is None or not (COMP / 'test.csv').exists():
    missing.append('the competition data (physionet-ecg-image-digitization)')
if HENGCK is None or not all((HENGCK / 'weight' / f).exists() for f in STAGE01_WEIGHTS):
    missing.append('the dataset hengck23/hengck23-demo-submit-physionet')
if MODELS is None or not all((MODELS / f).exists() for f in STAGE2_WEIGHTS):
    missing.append('the dataset takashisomeya/physionet-final-submission-models')
if missing:
    print('Content of /kaggle/input:')
    show_inputs()
    raise FileNotFoundError('Add these inputs to the notebook: ' + '; '.join(missing))

print('competition :', COMP)
print('stage 0/1   :', HENGCK)
print('stage 2     :', MODELS)
for f in STAGE2_WEIGHTS:
    print(f'   {f:42s} {(MODELS / f).stat().st_size / 2**20:6.0f} MB')


## Test set

`test.csv` lists the 12 leads of every ECG with the sampling rate and the number of rows to predict. A page has three rows of four leads (2.5 s each) and a 10 s lead II rhythm strip, so four series of 10 s describe the whole ECG. The row counts are used at the end to cut the series back into leads.

In [ ]:
test = pd.read_csv(COMP / 'test.csv', dtype={'id': str})
ids = test['id'].drop_duplicates().tolist()
# per ECG: [(lead, number_of_rows), ...] in test.csv order
layout = {k: [(lead, int(n)) for lead, n in zip(g['lead'], g['number_of_rows'])]
          for k, g in test.groupby('id', sort=False)}

print(f'{len(ids)} ECGs, {len(test)} rows in test.csv, {int(test["number_of_rows"].sum()):,} values to predict')
counts = lambda s: {int(k): int(v) for k, v in s.value_counts().items()}
print('leads per ECG   :', counts(test.groupby('id').size()))
print('sampling rate   :', counts(test.groupby('id')['fs'].first()))
print('lead II rows    :', counts(test.loc[test['lead'] == 'II', 'number_of_rows']))

try:
    import pyarrow.parquet as pq
    n_sample = pq.ParquetFile(COMP / 'sample_submission.parquet').metadata.num_rows
    print(f'sample_submission has {n_sample:,} rows, test.csv asks for {int(test["number_of_rows"].sum()):,}')
except Exception as exc:
    n_sample = None
    print('sample_submission.parquet not read:', exc)

display(test[test['id'] == ids[0]].set_index('lead'))


In [ ]:
small = cv2.imread(str(COMP / 'test' / f'{ids[0]}.png'), cv2.IMREAD_REDUCED_COLOR_4)
if small is not None:
    plt.figure(figsize=(11, 8))
    plt.imshow(cv2.cvtColor(small, cv2.COLOR_BGR2RGB))
    plt.title(f'test image {ids[0]} (shown at 1/4 size)')
    plt.axis('off')
    plt.show()


## Lead layout and metric

`split_leads` resamples the four series to the length of lead II, cuts each row at the lead boundaries given by `test.csv` and pads or trims every lead to its row count. The first block of the middle row is the beginning of the rhythm strip, so its length is the lead II length minus the other three blocks.

`snr_ecg` approximates the competition metric (best shift within 0.2 s, offset removed, powers summed over the 12 leads). It is only used for the check on training ECGs.

In [ ]:
LAYOUT = [['I', 'aVR', 'V1', 'V4'], ['II', 'aVL', 'V2', 'V5'], ['III', 'aVF', 'V3', 'V6']]


def split_leads(series, n_rows):
    # series: (4, T) in mV; rows 0-2 hold four leads each, row 3 is the lead II rhythm strip
    # n_rows: lead -> number of rows requested in test.csv
    total = int(n_rows['II'])
    if series.shape[1] != total:
        series = np.stack([resample(s, total) for s in series]).astype(np.float32)
    leads = {}
    for r, names in enumerate(LAYOUT):
        sizes = [int(n_rows[k]) for k in names]
        if names[0] == 'II':
            sizes[0] = max(0, sizes[0] - sum(sizes[1:]))
        for k, piece in zip(names, np.split(series[r], np.cumsum(sizes)[:-1])):
            leads[k] = piece
    leads['II'] = series[3]
    for k in leads:
        n = int(n_rows[k])
        leads[k] = np.pad(leads[k], (0, max(0, n - len(leads[k]))))[:n].astype(np.float32)
    return leads


def snr_ecg(truth, pred, fs, max_shift_s=0.2):
    max_shift = int(round(max_shift_s * fs))
    signal_power = noise_power = 0.0
    for lead, t in truth.items():
        p = pred[lead]
        n = min(len(t), len(p))
        t, p = np.nan_to_num(t[:n]), p[:n]
        best_lag, best_corr = 0, -np.inf
        for lag in range(-max_shift, max_shift + 1):
            a0, a1 = max(0, lag), min(n, n + lag)
            if a1 - a0 < 10:
                continue
            ta, pa = t[a0:a1], p[a0 - lag:a1 - lag]
            corr = np.dot(ta - ta.mean(), pa - pa.mean())
            if corr > best_corr:
                best_lag, best_corr = lag, corr
        a0, a1 = max(0, best_lag), min(n, n + best_lag)
        ta, pa = t[a0:a1], p[a0 - best_lag:a1 - best_lag]
        offset = np.mean(ta - pa)
        signal_power += np.sum(ta ** 2)
        noise_power += np.sum((ta - pa - offset) ** 2)
    return 10 * np.log10(signal_power / max(noise_power, 1e-12))


def read_truth(csv_path):
    # ground truth of a training ECG, with row counts in the test.csv convention
    df = pd.read_csv(csv_path)
    truth, n_rows = {}, {}
    for lead in [k for names in LAYOUT for k in names]:
        v = df[lead].to_numpy(dtype=float)
        if lead != 'II':
            v = v[~np.isnan(v)]            # short leads are NaN outside their 2.5 s block
        truth[lead], n_rows[lead] = v, len(v)
    return truth, n_rows


## Method details

**Rectified page.** After stages 0 and 1 every ECG sits on a 1700 x 2200 px canvas where 1 mV is 79 px and the 10 s of signal span columns 118 to 2080. The page is stretched horizontally by 5000 / 1962 so that one column equals one sample (2 ms). The zero lines of the four series lie at fixed rows, and rows above 416 hold no trace and are cropped for the whole-page networks.

**Stage 2 networks.**

- Whole-page networks see the cropped page (1280 x 5600 px) and predict one probability map per series.
- Lead-window networks see a strip of +-240 px (+-3 mV) around each zero line. The four strips go through one shared encoder and exchange features across the rows at four depths (`conv2d`, `shared_conv2d` or `conv3d` fusion).

**Ensemble.** The probability maps of all networks, and of the horizontally flipped page, are averaged per pixel. The 8 outermost rows of every window are dropped because predictions at the window border are less reliable. The weight map is the one of the 2nd place submission: whole-page predictions cover every row, window predictions only the rows of their window.

**Decoding.** The trace position in a column is the probability-weighted mean row (soft centroid), which gives sub-pixel positions. Rows are converted to mV with the zero line and 79 px/mV, and the 5000-point series are resampled with an FFT to the length given in `test.csv`.

## Code

The code is written to `/kaggle/working/ecg_infer/` so that the worker processes (one per GPU) can import it.

- `cc3d_compat.py`: scipy replacement for the two `connected-components-3d` calls of stages 0 and 1, used only if that package cannot be imported.
- `ecg_models.py`: the two stage 2 architectures.
- `ecg_pipeline.py`: stages 0 to 3 for one image, and the ensemble levels.
- `ecg_worker.py`: loop over images on one device, with the time budget control.

In [ ]:
%%writefile /kaggle/working/ecg_infer/cc3d_compat.py
"""Stand-in for the two cc3d calls used by the stage 0/1 code (2D masks only), based on scipy.

Used only if `connected-components-3d` cannot be imported (no prebuilt wheel for this Python version).
"""
import numpy as np
from scipy import ndimage as ndi

_EIGHT = np.ones((3, 3), dtype=bool)


def connected_components(mask, connectivity=8, **_):
    labels, _n = ndi.label(np.asarray(mask) != 0, structure=_EIGHT)
    return labels.astype(np.uint32, copy=False)


def statistics(labels):
    labels = np.asarray(labels)
    n = int(labels.max()) if labels.size else 0
    counts = np.bincount(labels.ravel(), minlength=n + 1).astype(np.uint32)
    centroids = np.zeros((n + 1, labels.ndim), dtype=np.float64)
    if n > 0:
        idx = np.nonzero(labels)
        lab = labels[idx]
        for d in range(labels.ndim):
            centroids[:, d] = np.bincount(lab, weights=idx[d], minlength=n + 1)
        centroids[1:] /= np.maximum(counts[1:], 1)[:, None]
    return {'voxel_counts': counts, 'centroids': centroids, 'bounding_boxes': None}


In [ ]:
%%writefile /kaggle/working/ecg_infer/ecg_models.py
"""Stage 2 segmentation networks: whole-image model and lead-window model.

Re-implementation of the two architectures of the 2nd place solution that needs only torch and timm.
Parameter names match the released checkpoints, so the weights load without remapping.
"""
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm


def _timm_name(name):
    """'tu-timm/tf_efficientnet_b7.ns_jft_in1k' -> 'tf_efficientnet_b7'.

    The pretrained tag only selects which weights timm would download; the checkpoints carry all weights.
    """
    name = name[3:] if name.startswith('tu-') else name
    name = name.split('/')[-1]
    return name.split('.')[0]


class TimmEncoder(nn.Module):
    """Feature pyramid from a timm backbone: [input, /2, /4, /8, /16, /32]."""

    def __init__(self, name):
        super().__init__()
        self.model = timm.create_model(
            _timm_name(name), pretrained=False, in_chans=3,
            features_only=True, out_indices=(0, 1, 2, 3, 4),
        )
        self.out_channels = [3] + list(self.model.feature_info.channels())

    def forward(self, x):
        return [x] + list(self.model(x))


class _SegNet(nn.Module):
    """Base class: ImageNet normalisation buffers named as in the checkpoints (D, mean, std)."""

    def __init__(self):
        super().__init__()
        self.register_buffer('D', torch.tensor(0))      # device marker
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).reshape(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).reshape(1, 3, 1, 1))

    def normalise(self, image):
        x = image.to(self.D.device).float() / 255
        return (x - self.mean) / self.std


# Whole-image model: timm encoder + CoordConv U-Net decoder

class CoordDecoderBlock(nn.Module):
    def __init__(self, in_channel, skip_channel, out_channel, scale=2):
        super().__init__()
        self.scale = scale
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channel + skip_channel + 2, out_channel, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channel),
            nn.ReLU(inplace=True),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channel, out_channel, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channel),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=self.scale, mode='nearest')
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        b, _, h, w = x.shape
        cx, cy = torch.meshgrid(
            torch.linspace(-2, 2, w, dtype=x.dtype, device=x.device),
            torch.linspace(-2, 2, h, dtype=x.dtype, device=x.device),
            indexing='xy',
        )
        coord = torch.stack([cx, cy], dim=1).reshape(1, 2, h, w).repeat(b, 1, 1, 1)
        x = torch.cat([x, coord], dim=1)
        return self.conv2(self.conv1(x))


class CoordUnetDecoder(nn.Module):
    def __init__(self, in_channel, skip_channel, out_channel):
        super().__init__()
        in_ch = [in_channel] + list(out_channel[:-1])
        self.block = nn.ModuleList(
            [CoordDecoderBlock(i, s, o, 2) for i, s, o in zip(in_ch, skip_channel, out_channel)]
        )

    def forward(self, feature, skip):
        d = feature
        for block, s in zip(self.block, skip):
            d = block(d, s)
        return d


class WholeNet(_SegNet):
    """Input (B,3,H,W) uint8 image, output (B,4,H,W) probabilities, one map per ECG row."""

    def __init__(self, encoder_name):
        super().__init__()
        self.encoder = TimmEncoder(encoder_name)
        enc = self.encoder.out_channels[2:]          # drop the input and the /2 stem feature
        dec = [256, 128, 64, 32]
        self.decoder = CoordUnetDecoder(enc[-1], enc[:-1][::-1] + [0], dec)
        # auxiliary heads are only used for training; kept so the checkpoint loads without remapping
        self.aux_pixel_heads = nn.ModuleList([nn.Conv2d(c + 1, 4, 1) for c in dec[:-1]])
        self.pixel = nn.Conv2d(dec[-1] + 1, 4, 1)

    def forward(self, image):
        _, _, H, W = image.shape
        x = self.normalise(image)
        f = self.encoder(x)[2:]
        last = self.decoder(f[-1], f[:-1][::-1] + [None])
        _, _, h, w = last.shape
        coordy = torch.arange(h, device=x.device).reshape(1, 1, h, 1).repeat(x.shape[0], 1, 1, w)
        coordy = coordy / (h - 1) * 2 - 1
        pixel = self.pixel(torch.cat([last, coordy], dim=1))
        if pixel.shape[2:] != (H, W):
            pixel = F.interpolate(pixel, size=(H, W), mode='bilinear', align_corners=False)
        return torch.sigmoid(pixel)


# Lead-window model: shared U-Net over the four rows with cross-row fusion

class UnetDecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels + skip_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, height, width, skip=None):
        x = F.interpolate(x, size=(height, width), mode='nearest')
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        return self.conv2(self.conv1(x))


class UnetDecoder(nn.Module):
    def __init__(self, encoder_channels, decoder_channels):
        super().__init__()
        enc = list(encoder_channels[1:])[::-1]
        in_ch = [enc[0]] + list(decoder_channels[:-1])
        skip_ch = list(enc[1:]) + [0]
        self.blocks = nn.ModuleList(
            [UnetDecoderBlock(i, s, o) for i, s, o in zip(in_ch, skip_ch, decoder_channels)]
        )

    def forward(self, features):
        shapes = [f.shape[2:] for f in features][::-1]
        feats = features[1:][::-1]
        x, skips = feats[0], feats[1:]
        for i, block in enumerate(self.blocks):
            h, w = shapes[i + 1]
            x = block(x, h, w, skips[i] if i < len(skips) else None)
        return x


class Conv3dBlock(nn.Module):
    def __init__(self, channels, stride, padding):
        super().__init__()
        self.conv = nn.Conv3d(channels, channels, (3, 3, 3), stride=stride, padding=padding,
                              padding_mode='replicate', bias=False)
        self.bn = nn.BatchNorm3d(channels)
        self.activation = nn.LeakyReLU(inplace=True)

    def forward(self, x):
        return self.activation(self.bn(self.conv(x)))


class CrossLeadFusion(nn.Module):
    """Mixes features of the four ECG rows and adds the result back to each row."""

    def __init__(self, channels, num_leads=4, fusion_type='conv2d', reduction_ratio=4, conv3d_depth=2):
        super().__init__()
        self.num_leads = num_leads
        self.fusion_type = fusion_type
        if fusion_type in ('conv2d', 'shared_conv2d'):
            red = channels // reduction_ratio

            def reduce():
                return nn.Sequential(nn.Conv2d(channels, red, 1, bias=False),
                                     nn.BatchNorm2d(red), nn.ReLU(inplace=True))

            if fusion_type == 'shared_conv2d':
                self.reduce_conv = reduce()
            else:
                self.reduce_convs = nn.ModuleList([reduce() for _ in range(num_leads)])
            self.mix_conv = nn.Sequential(
                nn.Conv2d(channels, channels, 3, padding=1, bias=False),
                nn.BatchNorm2d(channels), nn.ReLU(inplace=True),
                nn.Conv2d(channels, channels, 3, padding=1, bias=False),
                nn.BatchNorm2d(channels),
            )
        elif fusion_type == 'conv3d':
            stages = int(np.log2(num_leads))
            blocks = []
            for i in range(conv3d_depth):
                if i < stages:
                    blocks.append(Conv3dBlock(channels, (2, 1, 1), (0, 1, 1)))
                else:
                    blocks.append(Conv3dBlock(channels, (1, 1, 1), (1, 1, 1)))
            self.conv3d_blocks = nn.ModuleList(blocks)
        else:
            raise ValueError(f'unknown fusion_type {fusion_type}')

    def forward(self, x, batch_size):
        B = batch_size
        C, H, W = x.shape[1:]
        leads = x.view(B, self.num_leads, C, H, W)
        if self.fusion_type == 'conv3d':
            fused = leads.transpose(1, 2)
            for block in self.conv3d_blocks:
                if fused.shape[2] == 1:
                    break
                fused = block(fused)
            fused = fused.squeeze(2)
        else:
            if self.fusion_type == 'shared_conv2d':
                reduced = [self.reduce_conv(leads[:, i]) for i in range(self.num_leads)]
            else:
                reduced = [self.reduce_convs[i](leads[:, i]) for i in range(self.num_leads)]
            fused = self.mix_conv(torch.cat(reduced, dim=1))
        out = leads + fused.unsqueeze(1).expand(B, self.num_leads, C, H, W)
        return out.view(B * self.num_leads, C, H, W)


class LeadNet(_SegNet):
    """Input (B,4,3,H,W) uint8 windows, output (B,4,1,H,W) probabilities."""

    def __init__(self, encoder_name, fusion_type='conv2d', fusion_levels=(1, 2, 3, 4), num_leads=4):
        super().__init__()
        self.num_leads = num_leads
        self.fusion_levels = list(fusion_levels)
        self.encoder = TimmEncoder(encoder_name)
        self.decoder = UnetDecoder(self.encoder.out_channels, [256, 128, 64, 32, 16])
        self.fusion_modules = nn.ModuleDict({
            f'fusion_{lv}': CrossLeadFusion(self.encoder.out_channels[lv + 1], num_leads, fusion_type)
            for lv in self.fusion_levels
        })
        self.pixel_head = nn.Conv2d(16, 1, 1)

    def forward(self, image):
        B, L, C, H, W = image.shape
        x = self.normalise(image).view(B * L, C, H, W)
        feats = self.encoder(x)
        for lv in self.fusion_levels:
            feats[lv + 1] = self.fusion_modules[f'fusion_{lv}'](feats[lv + 1], B)
        pixel = self.pixel_head(self.decoder(feats))
        return torch.sigmoid(pixel.view(B, L, 1, H, W))


In [ ]:
%%writefile /kaggle/working/ecg_infer/ecg_pipeline.py
"""ECG image -> 12-lead signal pipeline.

Stage 0: orientation and perspective normalisation (hengck23, unchanged)
Stage 1: grid detection and rectification to a 1700 x 2200 page (hengck23, unchanged)
Stage 2: segmentation ensemble (2nd place models) and sub-pixel column centroid
"""
import os
import sys
import time
import traceback

import cv2
import numpy as np
import torch

HERE = os.path.dirname(os.path.abspath(__file__))
if HERE not in sys.path:
    sys.path.insert(0, HERE)

# Geometry of the rectified page
REF_H, REF_W = 1700, 2200            # canonical rectified page, pixels
# 10 s of signal span columns 118..2080 of the rectified page; the page is stretched
# horizontally so that they cover 5000 columns (one column = 2 ms).
X_SCALE = 5000 / (2080 - 118)
ADD_X = 1
IMG_H, IMG_W = REF_H, int(REF_W * X_SCALE) + ADD_X
T0, T1 = int(118 * X_SCALE) + ADD_X, int(2080 * X_SCALE) + ADD_X
N_COLS = T1 - T0

OFFSET = 416                          # rows above this carry no trace: cropped for the whole models
Y1, X1 = 1696, 5600
CROP_H = Y1 - OFFSET
WINDOW = 240                          # lead windows span +-240 px (+-3 mV) around the zero line
EDGE = 8                              # border rows of a window that are ignored in the ensemble
ZERO_MV = [703.5, 987.5, 1271.5, 1531.5]      # zero-volt row of each of the 4 series on the page
MV_TO_PIXEL = 79.0                    # 1 mV = 79 px on the page
ZERO_MV_TRIM = [z - OFFSET for z in ZERO_MV]

# Rows of the cropped page where each series window is added to the ensemble:
# (crop_start, crop_end, window_start, window_end)
REGIONS = []
for _z in ZERO_MV_TRIM:
    _top, _bot = int(_z) - WINDOW, int(_z) + WINDOW
    _w0, _w1 = EDGE, -EDGE
    if _bot > CROP_H:
        _w1 = -(_bot - CROP_H + EDGE)
        _bot = CROP_H
    REGIONS.append((_top + EDGE, _bot - EDGE, _w0, _w1))

# Stage 2 models (public leaderboard score of each one in the 2nd place write-up)
MODEL_SPECS = {
    'b7':    dict(kind='whole', encoder='tf_efficientnet_b7',  file='whole_b7_lb22.93.pth'),
    'v2l':   dict(kind='whole', encoder='tf_efficientnetv2_l', file='whole_v2_l_lb22.60.pth'),
    'b6a':   dict(kind='lead', encoder='tf_efficientnet_b6',  fusion='shared_conv2d', file='series_b6_shared_conv2d_lb23.10.pth'),
    'b6b':   dict(kind='lead', encoder='tf_efficientnet_b6',  fusion='shared_conv2d', file='series_b6_shared_conv2d_lb23.00.pth'),
    'v2l3d': dict(kind='lead', encoder='tf_efficientnetv2_l', fusion='conv3d',        file='series_v2_l_conv3d_lb22.92.pth'),
    'v2l2d': dict(kind='lead', encoder='tf_efficientnetv2_l', fusion='conv2d',        file='series_v2_l_conv2d_lb22.85.pth'),
}

# Cost of one forward pass in TFLOPs (counted with torch FlopCounterMode on a 1280x5600 page crop
# for the whole models and on four 480x5600 windows for the lead models).
MODEL_TFLOPS = {'b7': 1.94, 'v2l': 3.98, 'b6a': 2.38, 'b6b': 2.38, 'v2l3d': 6.31, 'v2l2d': 6.26}

# Ensemble levels. Level 0 is the full 2nd place submission (6 models, horizontal flip TTA).
# Higher levels are cheaper; the worker leaves level 0 only when the time budget requires it.
LEVELS = {
    0: dict(whole=['b7', 'v2l'], lead=['b6a', 'b6b', 'v2l3d', 'v2l2d'], flip=(False, True)),
    1: dict(whole=['b7', 'v2l'], lead=['b6a', 'b6b', 'v2l3d', 'v2l2d'], flip=(False,)),
    2: dict(whole=['b7'],        lead=['b6a', 'v2l3d'],                 flip=(False,)),
    3: dict(whole=[],            lead=['b6a'],                          flip=(False,)),
}


def level_cost(level):
    cfg = LEVELS[level]
    return sum(MODEL_TFLOPS[k] for k in cfg['whole'] + cfg['lead']) * len(cfg['flip'])


def models_for_levels(levels):
    keys = []
    for lv in levels:
        for k in LEVELS[lv]['whole'] + LEVELS[lv]['lead']:
            if k not in keys:
                keys.append(k)
    return keys


def load_weights(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=True)
    except Exception:
        return torch.load(path, map_location='cpu', weights_only=False)


def import_stage01(hengck_dir):
    """Import hengck23's stage 0/1 code. Falls back to a scipy stand-in if cc3d is unavailable."""
    try:
        import cc3d
    except Exception:
        import cc3d_compat
        sys.modules['cc3d'] = cc3d_compat
    if hengck_dir not in sys.path:
        sys.path.insert(0, hengck_dir)
    import stage0_common as s0c
    import stage1_common as s1c
    from stage0_model import Net as Net0
    from stage1_model import Net as Net1
    return s0c, s1c, Net0, Net1


def affine_rectify(normalised):
    """Cheap rectification, used only when grid detection fails.

    Stage 0 places the page on a fixed frame (page x 0.5818 + (74, 90)); the inverse affine warp
    brings its output close to the canonical 1700 x 2200 page.
    """
    s = 1280 / REF_W
    M = np.array([[s, 0, 74.0], [0, s, 90.0]], dtype=np.float32)
    return cv2.warpAffine(normalised, M, (REF_W, REF_H), flags=cv2.INTER_LINEAR | cv2.WARP_INVERSE_MAP,
                          borderMode=cv2.BORDER_REPLICATE)


class Digitizer:
    def __init__(self, hengck_dir, model_dir, device='cuda:0', use_amp=True, model_keys=None, log=print,
                 amp_dtype=torch.float16):
        self.device = torch.device(device)
        self.amp = bool(use_amp)
        self.amp_dtype = amp_dtype
        self.log = log
        self.s0c, self.s1c, Net0, Net1 = import_stage01(hengck_dir)

        self.net0 = self._load_stage01(Net0, f'{hengck_dir}/weight/stage0-last.checkpoint.pth')
        self.net1 = self._load_stage01(Net1, f'{hengck_dir}/weight/stage1-last.checkpoint.pth')

        from ecg_models import WholeNet, LeadNet
        keys = model_keys or list(MODEL_SPECS)
        self.models = {}
        for k in keys:
            spec = MODEL_SPECS[k]
            net = WholeNet(spec['encoder']) if spec['kind'] == 'whole' else LeadNet(spec['encoder'], spec['fusion'])
            net.load_state_dict(load_weights(f"{model_dir}/{spec['file']}"), strict=True)
            self.models[k] = net.to(self.device).eval()
            log(f'loaded {k:6s} {spec["file"]}')

        self._y = torch.arange(CROP_H, device=self.device, dtype=torch.float32).view(1, CROP_H, 1)
        self._zero_trim = torch.tensor(ZERO_MV_TRIM, dtype=torch.float64)

    def _load_stage01(self, cls, path):
        net = cls(pretrained=False)
        net.load_state_dict(load_weights(path)['state_dict'], strict=True)
        net.eval()
        net.output_type = ['infer']
        return net.to(self.device)

    def _autocast(self, stage01=False):
        # stage 0/1 post-processing converts outputs to numpy, which needs fp16 (not bf16) tensors
        enabled = self.amp and (self.device.type == 'cuda' or not stage01)
        return torch.autocast(device_type=self.device.type, dtype=self.amp_dtype, enabled=enabled)

    def _forward(self, net, batch):
        """Run a stage 0/1 net. Falls back to one sample at a time on CPU or after a GPU out-of-memory error."""
        if self.device.type == 'cuda':
            try:
                with self._autocast(True), torch.no_grad():
                    return net(batch)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
        images = batch['image']
        outs = []
        with self._autocast(True), torch.no_grad():
            for i in range(images.shape[0]):
                outs.append(net({'image': images[i:i + 1]}))
        return {k: torch.cat([o[k] for o in outs]) for k in outs[0]}

    def rectify(self, image):
        """image: RGB uint8. Returns (rectified 1700x2200 RGB uint8, route)."""
        s0c, s1c = self.s0c, self.s1c
        normalised = None
        try:
            batch = s0c.image_to_batch(image)
            out0 = self._forward(self.net0, batch)
            rotated, keypoint = s0c.output_to_predict(image, batch, out0)
            normalised, keypoint, _ = s0c.normalise_by_homography(rotated, keypoint)
        except Exception:
            self.log('stage0 failed:\n' + traceback.format_exc(limit=3))
        if normalised is None:
            return cv2.resize(image, (REF_W, REF_H), interpolation=cv2.INTER_AREA), 'resize'
        try:
            batch = {'image': torch.from_numpy(np.ascontiguousarray(normalised.transpose(2, 0, 1))).unsqueeze(0)}
            out1 = self._forward(self.net1, batch)
            grid_xy, _ = s1c.output_to_predict(normalised, batch, out1)
            return s1c.rectify_image(normalised, grid_xy), 'grid'
        except Exception:
            self.log('stage1 failed:\n' + traceback.format_exc(limit=3))
            return affine_rectify(normalised), 'affine'

    def _inputs(self, rectified):
        page = cv2.resize(rectified, (IMG_W, IMG_H), interpolation=cv2.INTER_LINEAR)
        img = torch.from_numpy(np.ascontiguousarray(page)).to(self.device).permute(2, 0, 1)   # 3,H,W uint8
        crop = img[:, OFFSET:Y1, :X1].unsqueeze(0).contiguous()                                # 1,3,1280,5600
        full = img[:, :Y1, :X1]
        wins = torch.zeros((4, 3, 2 * WINDOW, X1), dtype=torch.uint8, device=self.device)
        for i, z in enumerate(ZERO_MV):
            h0, h1 = int(z) - WINDOW, int(z) + WINDOW
            s0, s1 = max(0, h0), min(Y1, h1)
            wins[i, :, s0 - h0:s0 - h0 + (s1 - s0)] = full[:, s0:s1]
        return crop, wins.unsqueeze(0)                                                         # 1,4,3,480,5600

    def probability_maps(self, rectified, level):
        """Averaged probability maps (4, 1280, 5600) of the selected ensemble."""
        cfg = LEVELS[level]
        crop, wins = self._inputs(rectified)
        n_whole = len(cfg['whole']) * len(cfg['flip'])
        n_lead = len(cfg['lead']) * len(cfg['flip'])
        acc = torch.zeros((4, CROP_H, X1), dtype=torch.float32, device=self.device)
        with self._autocast(), torch.no_grad():
            # Horizontal flip = last axis (width) of the input, and of the (rows, H, W) output
            for k in cfg['whole']:
                for flip in cfg['flip']:
                    p = self.models[k](crop.flip(-1) if flip else crop)[0].float()
                    acc += p.flip(-1) if flip else p
            for k in cfg['lead']:
                for flip in cfg['flip']:
                    p = self.models[k](wins.flip(-1) if flip else wins)[0, :, 0].float()
                    if flip:
                        p = p.flip(-1)
                    for i, (a, b, c, d) in enumerate(REGIONS):
                        acc[i, a:b] += p[i, c:d]
        # Weight map of the 2nd place submission, shared by the four series: whole-model predictions
        # cover every row, lead-model predictions only the rows of their window (rows covered by two
        # windows get the lead weight twice).
        weight = torch.full((1, CROP_H, 1), float(n_whole), device=self.device)
        for a, b, _, _ in REGIONS:
            weight[0, a:b] += n_lead
        return acc / weight.clamp_min(1.0)

    def column_centroid(self, prob):
        """Sub-pixel y position of the trace in each of the 5000 columns -> series in mV."""
        sel = prob[:, :, T0:T1].double()
        den = sel.sum(1)
        num = (sel * self._y.double()).sum(1)
        y = (num / (den + 1e-8)).cpu()
        return ((self._zero_trim.view(4, 1) - y) / MV_TO_PIXEL).numpy().astype(np.float32)

    def digitize(self, image, level=0):
        t0 = time.perf_counter()
        rectified, route = self.rectify(image)
        t1 = time.perf_counter()
        prob = self.probability_maps(rectified, level)
        series = self.column_centroid(prob)
        t2 = time.perf_counter()
        return series, dict(route=route, t_rectify=round(t1 - t0, 2), t_segment=round(t2 - t1, 2))


def read_rgb(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise IOError(f'cannot read {path}')
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


In [ ]:
%%writefile /kaggle/working/ecg_infer/ecg_worker.py
"""Worker process: digitises ECG images on one device.

Usage: python ecg_worker.py config.json

Workers (one per GPU) share one job list; a worker claims an image by creating an empty file in the
claim directory. Each result is a (4, 5000) float32 array in mV: three lead rows plus the lead II strip.
The ensemble level (see ecg_pipeline.LEVELS) is chosen per image from the time that is left.
"""
import json
import os
import sys
import time
import traceback
from statistics import median

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')


def log(msg):
    print(time.strftime('%H:%M:%S'), msg, flush=True)


class LevelController:
    """Chooses the ensemble level for the next image.

    budget = time left / images left. The best level whose expected time per image fits the budget is
    used. A level that was never run is estimated from a measured level by the FLOP ratio, with a
    fixed share of the time (stage 0/1, image decoding) that does not scale with the ensemble.
    """
    FIXED_SHARE = 0.15

    def __init__(self, costs, warmup=1):
        self.costs = dict(costs)
        self.levels = sorted(self.costs)
        self.hist = {lv: [] for lv in self.levels}
        self.warmup = warmup            # the first image includes CUDA initialisation, ignore its time
        self.seen = 0

    def record(self, level, seconds):
        self.seen += 1
        if self.seen > self.warmup:
            self.hist[level].append(seconds)

    def ban(self, level):
        """Never choose this level again (it ran out of memory)."""
        if level in self.levels and len(self.levels) > 1:
            self.levels.remove(level)

    def estimate(self, level):
        recent = self.hist[level][-8:]
        if recent:
            return median(recent)
        known = [(len(v), lv) for lv, v in self.hist.items() if v]
        if not known:
            return None
        ref = max(known)[1]
        t = median(self.hist[ref][-8:])
        share = self.FIXED_SHARE
        return t * (share + (1 - share) * self.costs[level] / self.costs[ref])

    def choose(self, now, deadline, remaining):
        best = self.levels[0]
        if len(self.hist[best]) < 2:
            return best                  # measure the best level first
        budget = (deadline - now) / max(remaining, 1)
        for lv in self.levels:
            if self.estimate(lv) <= budget:
                return lv
        return self.levels[-1]


def claim(path):
    try:
        os.close(os.open(path, os.O_CREAT | os.O_EXCL | os.O_WRONLY))
        return True
    except FileExistsError:
        return False


ALIVE_WINDOW = 900      # a worker counts as running if its heartbeat file was touched in the last 15 minutes


def touch(path):
    try:
        with open(path, 'a'):
            os.utime(path, None)
    except OSError:
        pass


def workers_running(claim_dir):
    """Number of workers that share the job list (this one included), used to split the remaining images."""
    now, n = time.time(), 0
    for name in os.listdir(claim_dir):
        if name.startswith('alive_'):
            try:
                n += now - os.path.getmtime(os.path.join(claim_dir, name)) < ALIVE_WINDOW
            except OSError:
                pass
    return max(1, n)


def save_atomic(path, array):
    import numpy as np
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + '.tmp.npy'
    np.save(tmp, array)
    os.replace(tmp, path)


def main(cfg_path):
    cfg = json.load(open(cfg_path))
    import numpy as np
    import torch
    import cv2
    torch.set_num_threads(cfg['threads'])
    cv2.setNumThreads(cfg['threads'])
    from ecg_pipeline import Digitizer, LEVELS, level_cost, models_for_levels, read_rgb

    os.makedirs(cfg['claim_dir'], exist_ok=True)
    heartbeat = os.path.join(cfg['claim_dir'], f'alive_{os.getpid()}')
    touch(heartbeat)
    allowed = [lv for lv in sorted(LEVELS) if lv >= cfg.get('min_level', 0)]
    amp_dtype = torch.bfloat16 if cfg.get('amp_dtype') == 'bfloat16' else torch.float16
    t_load = time.time()
    dg = Digitizer(cfg['hengck_dir'], cfg['model_dir'], device=cfg['device'], use_amp=cfg['use_amp'],
                   model_keys=models_for_levels(allowed), amp_dtype=amp_dtype, log=log)
    log(f'models ready in {time.time() - t_load:.0f}s on {cfg["device"]}, levels {allowed}')

    ctrl = LevelController({lv: level_cost(lv) for lv in allowed})
    jobs = cfg['jobs']
    for job in jobs:
        job['claim'] = os.path.join(cfg['claim_dir'], job['id'])
    status = open(cfg['status_path'], 'a', buffering=1)
    n_done = 0

    for k, job in enumerate(jobs):
        if time.time() > cfg['hard_deadline']:
            log('hard deadline reached, stopping')
            break
        if os.path.exists(job['out']) or not claim(job['claim']):
            continue

        touch(heartbeat)
        canary = bool(job.get('canary'))
        if canary:
            level = ctrl.levels[0]
        else:
            ahead = sum(1 for j in jobs[k + 1:] if not os.path.exists(j['out']) and not os.path.exists(j['claim']))
            level = ctrl.choose(time.time(), cfg['soft_deadline'], 1 + ahead / workers_running(cfg['claim_dir']))

        t0 = time.time()
        rec = {'id': job['id'], 'level': level}
        try:
            image = read_rgb(job['path'])
            try:
                series, info = dg.digitize(image, level)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                ctrl.ban(level)
                level = next((lv for lv in ctrl.levels if lv > level), ctrl.levels[-1])
                rec['level'] = level
                log(f'out of memory, falling back to level {level}')
                series, info = dg.digitize(image, level)
            if not np.isfinite(series).all():
                series = np.nan_to_num(series, nan=0.0, posinf=0.0, neginf=0.0)
                info['nonfinite'] = True
            save_atomic(job['out'], series.astype(np.float32))
            rec.update(info, ok=True)
        except Exception:
            rec.update(ok=False, error=traceback.format_exc(limit=5))
            log(f'FAILED {job["id"]}\n{rec["error"]}')
        rec['sec'] = round(time.time() - t0, 2)
        if rec.get('ok'):
            ctrl.record(rec['level'], rec['sec'])
        status.write(json.dumps(rec) + '\n')
        n_done += 1
        if n_done % 25 == 0:
            log(f'{n_done} images done, level {rec["level"]}, last {rec["sec"]:.1f}s')
        if cfg['device'].startswith('cuda'):
            torch.cuda.empty_cache()

    log(f'worker finished, {n_done} images')
    open(cfg['done_path'], 'w').write('done')
    try:
        os.remove(heartbeat)
    except OSError:
        pass


if __name__ == '__main__':
    try:
        main(sys.argv[1])
    except Exception:
        log('worker crashed\n' + traceback.format_exc())
        sys.exit(1)


In [ ]:
sys.path.insert(0, str(WORK))
import ecg_pipeline as P

levels = pd.DataFrame([
    dict(level=lv,
         whole=', '.join(cfg['whole']) or '-',
         lead=', '.join(cfg['lead']),
         passes=len(cfg['flip']),
         tflops_per_ecg=round(P.level_cost(lv), 1))
    for lv, cfg in P.LEVELS.items()
]).set_index('level')
display(levels)
print(f'soft deadline {CONFIG["soft_deadline"] / 3600:.2f} h, hard deadline {CONFIG["hard_deadline"] / 3600:.2f} h after the start')


## Devices

Each GPU is probed with a small fp16 matrix product in a separate process, and every usable GPU gets its own worker process. The notebook kernel itself never uses the GPUs.

In [ ]:
def list_gpus():
    try:
        out = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, timeout=60).stdout
        found = [int(line.split(':')[0].split()[1]) for line in out.splitlines() if line.startswith('GPU ')]
    except Exception:
        found = []
    return found or list(range(torch.cuda.device_count()))


def probe_gpu(index):
    code = ("import torch; x = torch.randn(256, 256, device='cuda', dtype=torch.float16); "
            "(x @ x).float().mean().item(); p = torch.cuda.get_device_properties(0); "
            "print(p.name, round(p.total_memory / 2**30, 1), 'GB')")
    r = subprocess.run([sys.executable, '-c', code], capture_output=True, text=True, timeout=600,
                       env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(index)))
    lines = (r.stdout.strip() or r.stderr.strip()).splitlines()
    return r.returncode == 0, (lines[-1] if lines else '')


devices = []
for i in list_gpus():
    ok, info = probe_gpu(i)
    print(f'GPU {i}: {"ok" if ok else "unusable"} ({info})')
    if ok:
        devices.append(i)
if not devices:
    if not CONFIG['allow_cpu']:
        raise RuntimeError('No usable GPU. Switch the accelerator to GPU in the notebook settings.')
    devices = [None]
    print('running on CPU')


## Jobs

One job is one image and one output file with the four series. A few training ECGs with known ground truth go through the same code first (full ensemble), so their score in the log shows early whether the pipeline works.

In [ ]:
SERIES_DIR = WORK / 'series'
SERIES_DIR.mkdir(exist_ok=True)

test_ids = ids[:CONFIG['max_images']] if CONFIG['max_images'] else ids
jobs = [dict(id=i, path=str(COMP / 'test' / f'{i}.png'), out=str(SERIES_DIR / f'{i}.npy')) for i in test_ids]
absent = [j['id'] for j in jobs if not os.path.exists(j['path'])]
if absent:
    raise FileNotFoundError(f'{len(absent)} test images are missing, first ones: {absent[:5]}')


def find_canaries(n):
    found = []
    train_csv = COMP / 'train.csv'
    if n and train_csv.exists():
        for cid in pd.read_csv(train_csv, dtype=str, nrows=300).iloc[:, 0]:
            image = COMP / 'train' / cid / f'{cid}-0001.png'
            truth = COMP / 'train' / cid / f'{cid}.csv'
            if image.exists() and truth.exists():
                found.append(dict(id=f'canary_{cid}', path=str(image), out=str(SERIES_DIR / f'canary_{cid}.npy'),
                                  canary=True, truth=str(truth)))
            if len(found) == n:
                break
    return found


canaries = find_canaries(CONFIG['n_canary'])
all_jobs = canaries + jobs
print(f'{len(jobs)} test ECGs, {len(canaries)} training ECGs for the check')


## Running the workers

`launch` starts one worker per device. Workers share the job list and claim an image by creating an empty file, so the faster GPU takes more images and no image is processed twice. `wait` follows the status files, prints progress with an estimate of the remaining time, scores the training ECGs when they are done and stops workers that run past the deadline. Results are saved image by image, so a crashed worker loses at most the image it was working on.

In [ ]:
def read_status(paths):
    recs = []
    for p in paths:
        if os.path.exists(p):
            for line in open(p):
                try:
                    recs.append(json.loads(line))
                except ValueError:
                    pass                       # last line may be half written
    return recs


def tail(path, n=25):
    try:
        return ''.join(open(path).readlines()[-n:])
    except OSError:
        return ''


def launch(job_list, pass_no, min_level):
    claim_dir = WORK / f'claims_{pass_no}'
    shutil.rmtree(claim_dir, ignore_errors=True)
    workers = []
    for w, dev in enumerate(devices):
        tag = f'p{pass_no}_w{w}'
        cfg = dict(
            jobs=job_list, device='cpu' if dev is None else 'cuda:0',
            use_amp=True, amp_dtype='bfloat16' if dev is None else 'float16',
            threads=max(1, min(4, os.cpu_count() or 4) // len(devices)),
            hengck_dir=str(HENGCK), model_dir=str(MODELS),
            claim_dir=str(claim_dir), min_level=min_level,
            soft_deadline=T_START + CONFIG['soft_deadline'], hard_deadline=T_START + CONFIG['hard_deadline'],
            status_path=str(WORK / f'status_{tag}.jsonl'), done_path=str(WORK / f'done_{tag}'),
        )
        for stale in (cfg['status_path'], cfg['done_path']):
            if os.path.exists(stale):
                os.remove(stale)
        cfg_path = WORK / f'config_{tag}.json'
        cfg_path.write_text(json.dumps(cfg))
        env = dict(os.environ)
        if dev is not None:
            env['CUDA_VISIBLE_DEVICES'] = str(dev)      # the worker sees its GPU as cuda:0
        log_path = WORK / f'worker_{tag}.log'
        proc = subprocess.Popen([sys.executable, '-u', str(WORK / 'ecg_worker.py'), str(cfg_path)],
                                env=env, stdout=open(log_path, 'w'), stderr=subprocess.STDOUT, cwd=str(WORK))
        workers.append(dict(tag=tag, dev=dev, proc=proc, status=cfg['status_path'], log=str(log_path), reported=False))
        print(f'started worker {tag} on {"CPU" if dev is None else f"GPU {dev}"}')
    return workers


canary_done = False


def check_canaries():
    # score the training ECGs once all of them are done
    global canary_done
    if canary_done or not canaries or not all(os.path.exists(c['out']) for c in canaries):
        return
    canary_done = True
    scores = []
    try:
        for c in canaries:
            truth, n_rows = read_truth(c['truth'])
            leads = split_leads(np.load(c['out']), n_rows)
            scores.append(snr_ecg(truth, leads, n_rows['II'] / 10))
        print(f'[{elapsed() / 60:6.1f} min] check on {len(scores)} training ECGs: '
              f'SNR {", ".join(f"{s:.1f}" for s in scores)} dB, mean {np.mean(scores):.1f} dB')
        if np.mean(scores) < 8:
            print('WARNING: this is far below what the pipeline should reach, look at the worker logs.')
    except Exception as exc:
        print('check on training ECGs failed:', repr(exc))


def wait(workers, total, report_every=120, poll=10):
    history, last_report = [], 0.0
    while True:
        running = [w for w in workers if w['proc'].poll() is None]
        now = time.time()
        if now - last_report >= report_every or not running:
            recs = read_status([w['status'] for w in workers])
            done = sum(1 for r in recs if r.get('ok'))
            failed = len(recs) - done
            history.append((now, done))
            history = [h for h in history if now - h[0] <= 900]
            rate = (done - history[0][1]) / max(now - history[0][0], 1e-9)
            eta = f'{(total - done) / rate / 60:.0f} min' if rate > 0 else 'n/a'
            levels_used = dict(sorted(Counter(r['level'] for r in recs if r.get('ok')).items()))
            print(f'[{elapsed() / 60:6.1f} min] {done}/{total} done, {failed} failed, '
                  f'{rate * 60:.1f} img/min, eta {eta}, ensemble levels {levels_used}')
            last_report = now
        check_canaries()
        for w in workers:
            if w['proc'].poll() not in (None, 0) and not w['reported']:
                w['reported'] = True
                print(f'worker {w["tag"]} exited with code {w["proc"].returncode}:\n{tail(w["log"])}')
        if not running:
            return
        if elapsed() > CONFIG['hard_deadline'] + 600:
            print('deadline exceeded, stopping the workers')
            for w in running:
                w['proc'].kill()
            return
        time.sleep(poll)


In [ ]:
workers = launch(all_jobs, pass_no=1, min_level=CONFIG['min_level'])
wait(workers, total=len(all_jobs))
missing_jobs = [j for j in jobs if not os.path.exists(j['out'])]
print(f'[{elapsed() / 60:6.1f} min] first pass finished, {len(missing_jobs)} of {len(jobs)} test ECGs without result')


In [ ]:
# Second pass over the images that a crashed worker (e.g. CUDA error) left open.
if missing_jobs and elapsed() < CONFIG['hard_deadline'] - 900:
    healthy = [w['dev'] for w in workers if w['proc'].returncode == 0]
    devices = (healthy or devices)[:1]
    workers = launch(missing_jobs, pass_no=2, min_level=CONFIG['min_level'])
    wait(workers, total=len(missing_jobs))
    missing_jobs = [j for j in jobs if not os.path.exists(j['out'])]
    print(f'[{elapsed() / 60:6.1f} min] second pass finished, {len(missing_jobs)} test ECGs still without result')

if len(missing_jobs) == len(jobs):
    for log_file in sorted(WORK.glob('worker_*.log')):
        print(f'--- {log_file.name}\n{tail(log_file, 40)}')
    raise RuntimeError('No test ECG was digitised, see the worker logs above.')
check_canaries()


## Submission

The file is written ECG by ECG to keep memory small. An ECG without a result gets zeros. Values are cleaned of non-finite numbers and clipped to +-10 mV, so that a failure in one column cannot produce absurd values. Five decimals (1e-5 mV) are far below the noise level of the metric. Afterwards the ids are compared row by row with `sample_submission.parquet`.

In [ ]:
def write_submission(ecg_ids, path):
    tmp = str(path) + '.tmp'
    no_result, n_rows_written = [], 0
    with open(tmp, 'w') as f:
        f.write('id,value\n')
        for k, ecg in enumerate(ecg_ids):
            try:
                series = np.load(SERIES_DIR / f'{ecg}.npy')
            except OSError:
                series = np.zeros((4, 5000), np.float32)
                no_result.append(ecg)
            series = np.clip(np.nan_to_num(series), -10, 10)
            leads = split_leads(series, dict(layout[ecg]))
            for lead, n in layout[ecg]:
                values = leads[lead].tolist()
                f.write('\n'.join(f'{ecg}_{i}_{lead},{v:.5f}' for i, v in enumerate(values)))
                f.write('\n')
                n_rows_written += n
            if k and k % 500 == 0:
                print(f'{k}/{len(ecg_ids)} ECGs written')
    os.replace(tmp, path)
    print(f'{len(ecg_ids)}/{len(ecg_ids)} ECGs written, {n_rows_written:,} rows')
    return no_result, n_rows_written


def check_against_sample(path, n_written):
    # same ids in the same order as sample_submission.parquet
    pf = pq.ParquetFile(COMP / 'sample_submission.parquet')
    if pf.metadata.num_rows != n_written:
        raise ValueError(f'row count differs: submission {n_written:,}, sample {pf.metadata.num_rows:,}')
    different = seen = 0
    with open(path) as f:
        next(f)
        for batch in pf.iter_batches(batch_size=500_000, columns=['id']):
            expected = batch.column('id').to_pylist()
            got = [line.split(',', 1)[0] for line in islice(f, len(expected))]
            seen += len(got)
            different += sum(a != b for a, b in zip(expected, got))
        leftover = sum(1 for _ in f)
    if leftover or different or seen != pf.metadata.num_rows:
        raise ValueError(f'{different:,} ids differ from the sample, {leftover} rows are extra, '
                         f'{pf.metadata.num_rows - seen} rows are missing')


no_result, n_written = write_submission(test_ids, CONFIG['submission_path'])
if CONFIG['max_images'] or n_sample is None:
    print('partial run or no sample submission: id check skipped')
else:
    check_against_sample(CONFIG['submission_path'], n_written)
    print('ids and row count match sample_submission.parquet')
size_mb = os.path.getsize(CONFIG['submission_path']) / 2**20
print(f'{CONFIG["submission_path"]}: {size_mb:.0f} MB, {len(no_result)} ECGs filled with zeros')


## Run summary

In [ ]:
recs = read_status(sorted(str(p) for p in WORK.glob('status_*.jsonl')))
ok = [r for r in recs if r.get('ok') and not r['id'].startswith('canary_')]
open_ids = set(no_result)
errors = {r['id']: r for r in recs if not r.get('ok') and r['id'] in open_ids}
print(f'digitised {len(ok)} of {len(test_ids)} test ECGs in {elapsed() / 3600:.2f} h, {len(no_result)} written as zeros')
if ok:
    print('ensemble level  :', dict(sorted(Counter(r['level'] for r in ok).items())))
    print('page rectified  :', dict(Counter(r.get('route') for r in ok)), '(grid = full rectification, anything else is a fallback)')
    print('seconds / image : median %.1f, 95th percentile %.1f' % (np.median([r['sec'] for r in ok]),
                                                                np.percentile([r['sec'] for r in ok], 95)))
for r in list(errors.values())[:3]:
    print('error example   :', r['id'], '\n', r.get('error', '')[-600:])

shown = [i for i in test_ids if (SERIES_DIR / f'{i}.npy').exists()]
for ecg in ([shown[0], shown[len(shown) // 2], shown[-1]] if len(shown) >= 3 else shown):
    series = np.load(SERIES_DIR / f'{ecg}.npy')
    fig, axes = plt.subplots(4, 1, figsize=(14, 7), sharex=True)
    t = np.arange(series.shape[1]) / 500
    for r, ax in enumerate(axes):
        ax.plot(t, series[r], lw=0.8)
        ax.set_ylabel('mV')
        ax.grid(alpha=0.3)
        if r < 3:
            for j, name in enumerate(LAYOUT[r]):
                ax.text(j * 2.5 + 0.05, ax.get_ylim()[1], name, va='top', fontsize=9)
                if j:
                    ax.axvline(j * 2.5, ls=':', c='gray', lw=0.8)
        else:
            ax.text(0.05, ax.get_ylim()[1], 'II (rhythm strip)', va='top', fontsize=9)
    axes[-1].set_xlabel('seconds')
    fig.suptitle(f'digitised ECG {ecg}')
    plt.tight_layout()
    plt.show()

# remove intermediate files, keep logs and status files
shutil.rmtree(SERIES_DIR, ignore_errors=True)
for d in WORK.glob('claims_*'):
    shutil.rmtree(d, ignore_errors=True)
print(sorted(os.listdir('/kaggle/working')))
